# EDA - Pontos Cegos de Negócio

Este notebook endereça 5 hipóteses e análises gerenciais focado em levantar insights para a apresentação aos stakeholders. Trabalhamos exclusivamente 
com a visão pré-entrega e dados disponíveis até `order_approved_at`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configurações de estilo executivo
plt.style.use('default')
sns.set_theme(style="white", palette="muted")
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Carregar dados
df = pd.read_parquet('data/gold/olist_gold.parquet')
print(f"Base carregada com {df.shape[0]} linhas e {df.shape[1]} colunas.")

### 1. O Impacto da Complexidade do Pedido (Feature Mais Importante)

In [ ]:
# Criar categorias de quantidade de itens
def categorize_items(count):
    if count == 1:
        return "1 item"
    elif count == 2:
        return "2 itens"
    elif count <= 4:
        return "3 a 4 itens"
    else:
        return "5+ itens"

df['item_category'] = df['order_item_count'].apply(categorize_items)

# Calcular taxa de bad review
df_items = df.groupby('item_category', observed=True)['bad_review'].mean().reset_index()

# Ordenar categoricamente
order = ["1 item", "2 itens", "3 a 4 itens", "5+ itens"]
df_items['item_category'] = pd.Categorical(df_items['item_category'], categories=order, ordered=True)
df_items = df_items.sort_values('item_category')

fig, ax = plt.subplots(figsize=(8, 5))
cores = ['#d3d3d3', '#d3d3d3', '#d3d3d3', '#e74c3c'] 
sns.barplot(data=df_items, x='item_category', y='bad_review', palette=cores, ax=ax)

ax.set_title("Pedidos com mais de 5 itens dobram o risco de avaliação ruim", fontsize=14, loc='left', pad=15, weight='bold')
ax.set_xlabel("Quantidade de Itens no Pedido")
ax.set_ylabel("Taxa de Avaliações Ruins (1-2 Estrelas)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

# Rótulos de dados
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'bottom', 
                xytext = (0, 5), 
                textcoords = 'offset points',
                fontsize=11)

plt.tight_layout()
plt.show()

print("\nInsight de Negócio:")
print("- A complexidade logística cresce desproporcionalmente com a adição de itens. Pedidos grandes (5+ itens) têm taxa de erro e insatisfação significativamente maior.")
print("- Isso justifica visualmente 'order_item_count' como a feature mais importante detectada pelo SHAP no modelo XGBoost.")

### 2. Sazonalidade e Colapso Logístico (Série Temporal)

In [ ]:
# Garantir que a data é datetime
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])

# Agrupar por mes/ano
df['year_month'] = df['order_approved_at'].dt.to_period('M')

# Taxa de bad review mensal
df_time = df.groupby('year_month')['bad_review'].agg(['mean', 'count']).reset_index()

# Filtrar para evitar distorções do começo/fim da operação se contiverem poucos dados
df_time = df_time[df_time['count'] > 100]
df_time['year_month_dt'] = df_time['year_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 5))

# Linha principal
ax.plot(df_time['year_month_dt'], df_time['mean'], color='#34495e', linewidth=2.5, marker='o')

# Destacar Nov/Dez de 2017 (Black Friday e Natal)
bf_period = df_time[(df_time['year_month_dt'] >= '2017-11-01') & (df_time['year_month_dt'] <= '2017-12-31')]
ax.plot(bf_period['year_month_dt'], bf_period['mean'], color='#e74c3c', linewidth=2.5, marker='o')
ax.fill_between(df_time['year_month_dt'], 0, df_time['mean'], where=(df_time['year_month_dt'] >= '2017-11-01') & (df_time['year_month_dt'] <= '2017-12-31'), color='#e74c3c', alpha=0.2)

ax.set_title("O Colapso de Fim de Ano: Avaliações ruins disparam na Black Friday e Natal", fontsize=14, loc='left', pad=15, weight='bold')
ax.set_ylabel("Taxa de Avaliações Ruins (1-2 Estrelas)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

# Rótulos para os picos
for idx, row in bf_period.iterrows():
    ax.annotate(f'{row["mean"]:.1%}',
                (row['year_month_dt'], row['mean']),
                textcoords="offset points",
                xytext=(0,10),
                ha='center',
                color='#c0392b',
                weight='bold')

plt.tight_layout()
plt.show()

print("\nInsight de Negócio:")
print("- Os picos históricos de má avaliação coincidem com a sobrecarga da rede de distribuição em novembro e dezembro.")
print("- A malha logística cede sob pressão, mostrando que o risco aumenta com o mês do ano (sazonalidade) independentemente do vendedor ou rota isolada.")

### 3. O Efeito Pareto nos Vendedores Ineficientes

In [ ]:
# Agrupar por vendedor
df_sellers = df.groupby('seller_id').agg(
    total_orders=('order_id', 'count'),
    bad_reviews=('bad_review', 'sum')
).reset_index()

# Ordenar pelos piores vendedores absolutos
df_sellers = df_sellers.sort_values('bad_reviews', ascending=False)
df_sellers['cumulative_bad_reviews'] = df_sellers['bad_reviews'].cumsum()
df_sellers['cumulative_pct'] = df_sellers['cumulative_bad_reviews'] / df_sellers['bad_reviews'].sum()

# Quantos % de vendedores representam cada parcela
df_sellers['seller_rank'] = range(1, len(df_sellers) + 1)
df_sellers['seller_pct'] = df_sellers['seller_rank'] / len(df_sellers)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_sellers['seller_pct'], df_sellers['cumulative_pct'], color='#e74c3c', linewidth=2.5)

# Linha de 80% (Pareto)
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.7)

# Encontrar o ponto dos 80%
pareto_point = df_sellers[df_sellers['cumulative_pct'] >= 0.8].iloc[0]
pct_sellers_80 = pareto_point['seller_pct']

ax.axvline(pct_sellers_80, color='gray', linestyle='--', alpha=0.7)
ax.scatter(pct_sellers_80, 0.8, color='#c0392b', zorder=5, s=100)
ax.text(pct_sellers_80 + 0.02, 0.70, f'{pct_sellers_80:.1%} dos vendedores\ncausam 80% das\navaliações ruins', fontsize=11, color='#c0392b')

ax.set_title("Efeito Pareto: Minoria de vendedores destrói a percepção da marca", fontsize=14, loc='left', pad=15, weight='bold')
ax.set_xlabel("% Total de Vendedores na Base")
ax.set_ylabel("% Acumulado de Avaliações Ruins")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\nInsight de Negócio:")
print(f"- Aproximadamente {pct_sellers_80:.1%} dos piores vendedores são responsáveis por 80% de toda a insatisfação da plataforma.")
print("- Um dashboard de score para descredenciar ou auditar essa fatura pequena de 'bad actors' reduziria violentamente as notas negativas.")

### 4. Correção da Análise de Frete

In [ ]:
# Criar bins para freight_ratio (assumindo que a feature existe com base na EDA anterior)
if 'freight_ratio' not in df.columns and 'freight_value' in df.columns and 'price' in df.columns:
    df['freight_ratio'] = df['freight_value'] / (df['freight_value'] + df['price'])

if 'freight_ratio' in df.columns:
    bins = [0, 0.1, 0.2, 0.3, float('inf')]
    labels = ['0-10%', '11-20%', '21-30%', '>30%']
    df['freight_ratio_bin'] = pd.cut(df['freight_ratio'], bins=bins, labels=labels, right=True)
    
    df_freight = df.groupby('freight_ratio_bin', observed=True)['bad_review'].mean().reset_index()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=df_freight, x='freight_ratio_bin', y='bad_review', color='#3498db', ax=ax)
    
    # Destacar a barra com maior frete
    ax.patches[-1].set_facecolor('#e74c3c')
    
    ax.set_title("Frete Caro = Intolerância: Frete acima de 30% inflaciona o risco", fontsize=14, loc='left', pad=15, weight='bold')
    ax.set_xlabel("Custo do Frete em Relação ao Valor do Pedido (freight_ratio)")
    ax.set_ylabel("Taxa de Avaliações Ruins (1-2 Estrelas)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
    
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1%}', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'bottom', 
                    xytext = (0, 5), 
                    textcoords = 'offset points',
                    fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    print("\nInsight de Negócio:")
    print("- A expectativa do cliente muda de acordo com o peso do frete no bolso.")
    print("- Quanto maior o percentual pago em frete, menor a tolerância para atrasos ou deslizes logísticos, aumentando drasticamente o rigor da avaliação.")
else:
    print("Coluna de freight_ratio ou price/freight_value não foi encontrada para a análise.")

### 5. Correção da Análise de Categorias de Produtos

In [ ]:
# Analisar o viés de produto
if 'product_category_name' in df.columns:
    df_cat = df.groupby('product_category_name').agg(
        total_orders=('order_id', 'count'),
        bad_review_rate=('bad_review', 'mean')
    ).reset_index()

    # Prevenir viés de pequenos volumes
    df_cat = df_cat[df_cat['total_orders'] >= 500]

    # Top 10 piores taxas percentuais
    top_10_cats = df_cat.sort_values('bad_review_rate', ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Criar gradiente de cores
    sns.barplot(data=top_10_cats, x='bad_review_rate', y='product_category_name', palette='Reds_r', ax=ax)

    ax.set_title("As 10 Categorias Mais Críticas: Analisadas por Proporção (>500 pedidos)", fontsize=14, loc='left', pad=15, weight='bold')
    ax.set_xlabel("Taxa de Avaliações Ruins (1-2 Estrelas)")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))

    # Rótulos nas barras
    for p in ax.patches:
        ax.annotate(f'{p.get_width():.1%}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha = 'left', va = 'center', 
                    xytext = (5, 0), 
                    textcoords = 'offset points',
                    fontsize=11)

    plt.tight_layout()
    plt.show()

    print("\nInsight de Negócio:")
    print("- Focando na proporção ao invés de contagem absoluta isolamos fatores logísticos difíceis.")
    print("- Categorias complexas (Móveis de escritório, telefonia) apresentam mais frustração, seja por caixas grandes sujeitas a avarias ou expectativas de integridade muito altas.")
else:
    print("Coluna 'product_category_name' não encontrada no dataset.")